# 05 — Eşik Kalibrasyonu (F3 ADIM 3)

**Metodolojik tuzak:** Gerçek veride etiket yoktur, dolayısıyla gerçek veriden
AUC hesaplanamaz. İş bölümü:

* **AUC / ROC ve K-11 kararı** → bozulmalı sentetik (etiket biliniyor) — `tools/ab_stage2.py`
* **Eşik değerleri** → gerçek veri skor dağılımları:
  * Katı = negatif kontrollerin pratikte hiç geçemediği nokta
  * Orta = şablon sorgularının ilk 12'sinin rahat geçtiği nokta
  * Esnek = pozitif kuyruğun tükendiği nokta

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 200)
OUT = Path("notebooks/output"); OUT.mkdir(parents=True, exist_ok=True)
TF = "1d"

from tools.f3_measure import calibrate_thresholds, MEANINGLESS_KINDS
from engine.config import TOLERANCE_THRESHOLDS, STAGE2_METRIC

dagilim = pd.read_json("data/f3/score_distribution.json")
kal = calibrate_thresholds(dagilim)
print("aktif metrik:", STAGE2_METRIC)
print("anlamsız kabul edilen sorgular:", MEANINGLESS_KINDS)
print(json.dumps(kal, indent=1, ensure_ascii=False))
print("\nconfig.py'deki değerler:", TOLERANCE_THRESHOLDS)

aktif metrik: hybrid
anlamsız kabul edilen sorgular: ('white_noise', 'whipsaw')
{
 "thresholds": {
  "strict": 0.0172,
  "medium": 0.0273,
  "flexible": 0.0836
 },
 "raw": {
  "strict": 0.0172,
  "medium": 0.0273,
  "flexible": 0.0836
 },
 "negative_floor": Infinity,
 "ceiling": Infinity,
 "capped": [],
 "n_negative": 0,
 "n_template": 14
}

config.py'deki değerler: {'strict': 0.0172, 'medium': 0.0273, 'flexible': 0.0836}


In [2]:
# K-11 A/B: sentetik, bozulmalı pozitiflerle 3 metrik x 4 uzunluk x 3 gürültü
ab = json.loads(Path("data/f3/ab_stage2.json").read_text(encoding="utf-8"))
satir = pd.DataFrame(ab["rows"])
ozet = satir.pivot_table(index=["noise", "length"], columns="metric", values="auc").round(4)
ozet["kazanan"] = ozet.idxmax(axis=1)
ozet

metric          ddtw     dtw  hybrid kazanan
noise length                                
0.01  24      0.9852  0.9893  0.9919  hybrid
      40      0.9787  0.9801  0.9854  hybrid
      64      0.9613  0.9700  0.9760  hybrid
      100     0.9487  0.9559  0.9647  hybrid
0.02  24      0.9478  0.9573  0.9633  hybrid
      40      0.9199  0.9257  0.9364  hybrid
      64      0.8724  0.9011  0.9131  hybrid
      100     0.8425  0.8629  0.8773  hybrid
0.04  24      0.8604  0.8601  0.8712  hybrid
      40      0.7906  0.8237  0.8348  hybrid
      64      0.7352  0.7755  0.7880  hybrid
      100     0.6370  0.7460  0.7525  hybrid

In [3]:
fig, ax = plt.subplots(figsize=(9, 4))
for metrik, g in satir.groupby("metric"):
    egri = g.groupby("length").auc.mean()
    ax.plot(egri.index, egri.values, "o-", label=metrik)
ax.set_xlabel("pencere uzunluğu (bar)"); ax.set_ylabel("AUC")
ax.set_title("K-11 A/B — pencere uzunluğuna göre kırılım (ortalama)")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT / "k11_ab_uzunluk.png", dpi=110)
satir.groupby("metric").auc.mean().round(4)

metric
ddtw      0.8733
dtw       0.8956
hybrid    0.9045
Name: auc, dtype: float64

In [4]:
# Eşiklerin görsel yeri
sablon = dagilim[dagilim.tur == "sablon"]
negatif = dagilim[dagilim.tur == "negatif"].copy()
negatif["tip"] = negatif.sorgu.str.rsplit("_", n=1).str[0]
anlamsiz = negatif[negatif.tip.isin(MEANINGLESS_KINDS)].en_iyi.replace([np.inf], np.nan).dropna()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sablon.top12, bins=14, alpha=0.8, label="şablon — 12. sonuç", color="#2b7a78")
if len(anlamsiz):
    ax.hist(anlamsiz, bins=8, alpha=0.6, label="anlamsız sorgu — en iyi", color="#c94f4f")
for ad, v in TOLERANCE_THRESHOLDS.items():
    ax.axvline(v, ls="--", label=f"{ad} = {v}")
ax.set_xlabel("normalize mesafe"); ax.set_title("Eşiklerin pozitif ve negatif dağılımlar arasındaki yeri")
ax.legend(); fig.tight_layout(); fig.savefig(OUT / "esik_kalibrasyon.png", dpi=110)

In [5]:
# Doğrulama: hangi kademede kim geçiyor?
for ad, v in TOLERANCE_THRESHOLDS.items():
    gecen_sablon = int((sablon.top12 <= v).sum())
    gecen_anlamsiz = int((anlamsiz <= v).sum()) if len(anlamsiz) else 0
    print(f"{ad:9s} {v:.4f} -> ilk 12'si tamamen geçen şablon: {gecen_sablon}/14, "
          f"eşiği geçen anlamsız sorgu: {gecen_anlamsiz}/{len(anlamsiz)}")

strict    0.0172 -> ilk 12'si tamamen geçen şablon: 4/14, eşiği geçen anlamsız sorgu: 0/0
medium    0.0273 -> ilk 12'si tamamen geçen şablon: 7/14, eşiği geçen anlamsız sorgu: 0/0
flexible  0.0836 -> ilk 12'si tamamen geçen şablon: 13/14, eşiği geçen anlamsız sorgu: 0/0
